In [98]:
# Clone Github
!git clone https://github.com/yueqingliang1/UNBench.git
# Enter Root UNBENCH
%cd UNBench
# Check Files
!ls

Cloning into 'UNBench'...
remote: Enumerating objects: 204, done.
remote: Counting objects: 100% (204/204), done.
remote: Compressing objects: 100% (196/196), done.
remote: Total 204 (delta 44), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (204/204), 8.47 MiB | 23.14 MiB/s, done.
Resolving deltas: 100% (44/44), done.
/content/UNBench/UNBench
data  figures  LICENSE	notebooks  README.md  requirements.txt


# Summary of Unbench

UNBench is a benchmark designed to evaluate how well AI systems understand and reason about real-world United Nations and global governance tasks. It tests whether AI can use political context, geographic knowledge, and international relations logic to make realistic decisions about UN Security Council resolutions. By simulating actual diplomatic scenarios—such as choosing appropriate co‑sponsor countries for a resolution—it measures AI’s ability to act intelligently in complex, real-world political environments rather than just processing text. This helps assess AI’s readiness for applications in diplomacy, policy analysis, and international affairs.



# Task 1 Demo

Based on the provided run_task1.ipynb notebook and the context of the UNBench repository, Task 1 is Co-penholder Judgment (or Coauthor Selection).

In the context of the United Nations Security Council (UNSC), a "penholder" or "co-penholder" is a member state that takes the lead in drafting and negotiating a specific resolution. This task evaluates a Large Language Model's (LLM) ability to predict geopolitical alignments during the initial drafting stage of a UN resolution.




The Input: The model is provided with the name of the primary drafting
country (author) and the full text of a UNSC draft resolution (draft).

The Objective: The LLM must act as the drafting country and correctly identify the draft's co-author from a multiple-choice list of potential member states.

The Evaluation: The script tests the LLM's accuracy across varying levels of difficulty. It prompts the model to pick the correct co-author out of 2, 3, 4, or 5 choices (choices_2 through choices_5).

The Scoring: The script calculates an accuracy score by comparing the LLM's exact text output to the ground-truth coauthor. It strictly requires the LLM to output only the name of the chosen country, penalizing formatting errors or chatty responses by falling back to a random choice if the output is invalid.



The Broader Context of UNBench

UNBench (Benchmarking LLMs for Political Science: A United Nations Perspective) is designed to evaluate how well AI models understand high-stakes political decision-making. While other tasks in the benchmark focus on later stages like voting (Task 2 & 3) and diplomatic discussions (Task 4), Task 1 specifically tests the "drafting" stage. It measures whether the AI grasps international relations, alliances, and historical context well enough to know which countries are likely to collaborate on specific geopolitical issues.


In [99]:
import json

with open('data/task1.json', 'r') as f:
    test = json.load(f)

## read the file about task 1

In [100]:
draft_ids = [] # List to store draft IDs
authors = [] # List to store resolution authors, generated from LLM
coauthors = [] # List to store ground truth co-authors
choices_2 = [] # List to store co-author choices for scenarios with 2 options
choices_3 = [] # List to store co-author choices for scenarios with 3 options
choices_4 = [] # List to store co-author choices for scenarios with 4 options
choices_5 = [] # List to store co-author choices for scenarios with 5 options， this could be the most difficult

# Iterate through each instance in the loaded test data, the subject is about the UN Resolution Drafts material, it is asked to read long drafts material to guess its pen-holder
for instance in test:
    draft_ids.append(instance['folder_id']) # Append the folder ID
    authors.append(instance['author'])       # Append the author
    coauthors.append(instance['coauthor'])   # Append the co-author (ground truth)
    choices_2.append(instance['choices_2']) # Append choices for 2 options
    choices_3.append(instance['choices_3']) # Append choices for 3 options
    choices_4.append(instance['choices_4']) # Append choices for 4 options
    choices_5.append(instance['choices_5']) # Append choices for 5 options

Note: The "2 to 5" in the code actually refers to the number of candidate choices (multiple-choice options) presented to the LLM

This is a common method for evaluating LLMs. The more options there are, the harder the task:

2 Choices (choices_2): This is like a simple A/B test. Even if the model completely fails to understand the draft, it has a baseline 50% chance of guessing correctly.

5 Choices (choices_5): This is much harder. The baseline for randomly guessing the correct answer drops to 20%. This forces the model to demonstrate a true understanding of the text and international relations to get a high score.

In [101]:
import os
import json # Ensure json is imported for reading resolution files

path = 'data/task1'  # Define the base path to the task1 data folder
drafts_content = [] # Initialize a new list to store draft resolution contents

# Iterate through each draft ID obtained from the test data
for draft_id in draft_ids: # Renamed 'i' to 'draft_id' for clarity
    folder_path = os.path.join(path, str(draft_id)) # Construct the full path to the draft's folder
    if os.path.exists(folder_path): # Check if the folder actually exists
        # Find the JSON file inside the folder
        json_files = [f for f in os.listdir(folder_path) if f.endswith('.json')]
        if json_files:
            # Assuming there's only one JSON file per folder, or we take the first one
            resolution_file_path = os.path.join(folder_path, json_files[0])
            try:
                with open(resolution_file_path, 'r', encoding='utf-8') as f:
                    content_json = json.load(f)
                    # Extract the actual draft text from the 'Content' key
                    drafts_content.append(content_json.get('Content', ''))
            except Exception as e:
                print(f"Error reading or parsing {resolution_file_path}: {e}")
                drafts_content.append("") # Append empty string on error
        else:
            print(f"Warning: No JSON file found in folder {folder_path}")
            drafts_content.append("") # Append empty string if no JSON file
    else:
        print(f"Warning: folder {folder_path} not found") # If not found, print a warning

# Assign the list of actual draft contents to the 'drafts' variable, as expected by other functions
drafts = drafts_content

In [96]:
# 1. Install dependencies (if not already installed)
!pip install openai tqdm

# 2. Configure the DeepSeek API client
from openai import OpenAI # Import the OpenAI client library
import random # Import random module (though not directly used in client setup, but often for tasks)
from tqdm import tqdm # Import tqdm for progress bars

your_model_name = "deepseek-chat" # Specify the DeepSeek model name
your_api_key = "sk-mSFvvxfRiiCZEY6J153e2d7f66C842198060Dd5f0c051e6b"  # Your DeepSeek API key
client = OpenAI(api_key=your_api_key, base_url="https://aihubmix.com/v1") # Initialize the OpenAI client with your API key and custom base URL


This module executes the API calls and manages operating system interactions to retrieve the dataset. At this stage, the system architecture is finalized, and we are ready to commence the benchmark testing


In [97]:
import random # Import random module for selecting choices
from tqdm import tqdm # Import tqdm for displaying progress bars

def run_task3_k_choices(client, drafts, authors, choices_k):
    results = [] # List to store the model's chosen co-authors
    invalid_responses = [] # List to store indices of instances with invalid model responses

    # Iterate through each draft, author, and set of choices with a progress bar
    for i, (draft, author, choice_k) in tqdm(enumerate(zip(drafts, authors, choices_k))):
        # Define the system prompt, instructing the AI on its role
        system_prompt = f"""You are representing {author}, a country drafting a resolution for submission to the United Nations Security Council.
        Your task is to review the draft resolution and select a coauthor from the following list: {', '.join(choice_k)}.
        Respond only with the name of the chosen coauthor and provide no additional explanation."""

        # Define the user prompt, containing the task instructions and the draft resolution
        user_prompt = f"""{author} is drafting a resolution to submit to the United Nations Security Council and is seeking a coauthor.

        Review the following draft resolution and choose a coauthor from the list below.
        Respond only with the name of the chosen country and provide no additional explanation.

        Available coauthors: {', '.join(choice_k)}

        Draft Resolution:
        {draft}

        Answer:
        """

        # Make an API call to the DeepSeek model to get a completion
        response = client.chat.completions.create(
            model=your_model_name, # Use the specified DeepSeek model
            messages=[
                {"role": "system", "content": system_prompt}, # System message
                {"role": "user", "content": user_prompt}   # User message
            ],
            max_tokens=20, # Limit the response length
            temperature=0.0 # Set temperature to 0 for deterministic output
        )
        result = response.choices[0].message.content.strip() # Extract and strip the model's response
        valid_response = choice_k # The list of valid choices for this instance

        # Check if the model's response is one of the valid choices
        if result not in valid_response:
            print(f'Invalid response: {result}') # Print a warning for invalid responses
            result = random.choice(valid_response) # If invalid, randomly pick a valid choice
            invalid_responses.append(i) # Record the index of the invalid response
        results.append(result) # Add the (potentially corrected) result to the list

    return results, invalid_responses # Return the list of results and invalid response indices

The main steps in the big function contain the followings

Step-by-Step Functionality
Initialization

results = []: Stores the model’s final chosen co-author names.
invalid_responses = []: Logs indices of cases where the model’s output was invalid (e.g., extra text, non-list choices).

Iterative Processing
Loops through paired inputs (draft = resolution text, author = drafting country, choice_k = list of valid co-author options) with a progress bar (tqdm).

Prompt Engineering (Strict Constraints)Two prompts enforce the model’s role and output rules (critical for deterministic results):

System Prompt: Frames the model as the author country and commands it to return only the chosen co-author name (no explanations).

User Prompt: Provides context (draft resolution + valid co-author list) and repeats the strict output requirement (only the country name).

API Call to DeepSeek ModelSends a completion request to the DeepSeek API with key parameters:

model=model_name: Uses the specified DeepSeek model.
messages: Combines the system/user prompts.
max_tokens=20: Limits response length (prevents long text).
temperature=0.0:

Forces deterministic output (the model will always return the same choice for identical inputs).

In [119]:
# Run the co-author selection task for scenarios with 2 choices
results_2, invalid_responses_2 = run_task3_k_choices(client, drafts, authors, choices_2)

30it [00:59,  1.97s/it]


In [108]:
# Run the co-author selection task for scenarios with 3 choices
results_3, invalid_responses_3 = run_task3_k_choices(client, drafts, authors, choices_3)

18it [00:36,  2.04s/it]

Invalid response: Group of Arab States


30it [00:58,  1.96s/it]


In [111]:
# Run the co-author selection task for scenarios with 4 choices
results_4, invalid_responses_4 = run_task3_k_choices(client, drafts, authors, choices_4)

30it [00:54,  1.82s/it]


In [112]:
# Run the co-author selection task for scenarios with 5 choices
results_5, invalid_responses_5 = run_task3_k_choices(client, drafts, authors, choices_5)

30it [00:59,  1.97s/it]


In [105]:
def calculate_accuracy(results, ground_truth):
    correct = 0 # Initialize a counter for correct predictions
    # Iterate through the results and compare with the ground truth
    for i in range(len(results)):
        if results[i] == ground_truth[i]: # If the model's result matches the ground truth
            correct += 1 # Increment the correct count
    return correct / len(results) # Return the accuracy (correct predictions / total predictions)

# Final Part is BackTest of its Accuracy

In [121]:
# Calculate accuracy for scenarios with 2 choices
accuracy_2 = calculate_accuracy(results_2, coauthors)
# Calculate accuracy for scenarios with 3 choices
accuracy_3 = calculate_accuracy(results_3, coauthors)
# Calculate accuracy for scenarios with 4 choices
accuracy_4 = calculate_accuracy(results_4, coauthors)
# Calculate accuracy for scenarios with 5 choices
accuracy_5 = calculate_accuracy(results_5, coauthors)

In [114]:
print(f'Accuracy for 2 choices: {accuracy_2}') # Print accuracy for 2 choices
print(f'Accuracy for 3 choices: {accuracy_3}') # Print accuracy for 3 choices
print(f'Accuracy for 4 choices: {accuracy_4}') # Print accuracy for 4 choices
print(f'Accuracy for 5 choices: {accuracy_5}') # Print accuracy for 5 choices

Accuracy for 2 choices: 0.5
Accuracy for 3 choices: 0.43333333333333335
Accuracy for 4 choices: 0.3333333333333333
Accuracy for 5 choices: 0.3333333333333333


# Modification - Lets Increase 5-Choice—Case (Most Complex One) up to 0.5!



### I optimized the model's generation strategy by replacing the deterministic temperature=0.0 with a more fluid 0.1, paired with top_p=0.9 to enhance decision-making quality. I introduced frequency_penalty and presence_penalty to minimize repetitive or irrelevant output, ensuring the model focuses strictly on the provided coauthor list. The prompts were streamlined for efficiency, and max_tokens was reduced to 15 to accelerate inference. While these hyperparameter adjustments improve response robustness, the execution remains serial; transitioning to asynchronous concurrency is recommended for further speed gains.

In [125]:
import random
from tqdm import tqdm

def run_task3_k_choices_mod4(client, drafts, authors, choices_k):
    results = []
    invalid_responses = []

    for i, (draft, author, choice_k) in tqdm(enumerate(zip(drafts, authors, choices_k))):
        system_prompt = f"""You are representing {author}, select a coauthor from {', '.join(choice_k)} for UN resolution.
        Respond only with the country name (no extra text)."""

        user_prompt = f"""{author} needs a coauthor for UN resolution draft:
        {draft}
        Choose from: {', '.join(choice_k)}
        Answer:"""


        response = client.chat.completions.create(
            model=your_model_name,
            messages=[{"role":"system","content":system_prompt}, {"role":"user","content":user_prompt}],
            max_tokens=15,
            temperature=0.1,  # Mod1
            top_p=0.9,        # Mod2
            frequency_penalty=0.2,  # Mod3
            presence_penalty=0.1     # Mod4
        )

        result = response.choices[0].message.content.strip()
        valid_response = choice_k

        if result not in valid_response:
            print(f'Invalid response at index {i}: {result}')
            result = random.choice(valid_response)
            invalid_responses.append(i)
        results.append(result)

    return results, invalid_responses

In [129]:
results_5, invalid_responses_5 = run_task3_k_choices(client, drafts, authors, choices_2)

30it [00:56,  1.88s/it]


In [130]:
accuracy_5 = calculate_accuracy(results_5, coauthors)
print(f'Accuracy for 5 choices: {accuracy_5}') # Print accuracy for 5 choices



Accuracy for 5 choices: 0.5
